# 04 - Gold Layer
## 02 - Gold Validation

### Goal
- validate all gold tables using SQL
- row counts, null checks, negative checks, distribution checks, summary checks

In [0]:
%run ../01_setup/00_config

In [0]:
import logging
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

## Row count checks - all gold tables

In [0]:
all_tables = [
    gold_regional_operations_table,
    gold_market_volatility_table,
    gold_market_summary_table,
    gold_grid_summary_table,
    gold_weather_impact_table,
    gold_operational_kpi_table,
]

for table in all_tables:
    try:
        count = spark.table(table).count()
        if count > 0:
            log.info(f"[ROWS]    {table} has {count} rows.")
        else:
            raise ValueError(f"[ROWS]    {table} is empty.")
    except AnalysisException:
        raise ValueError(f"[MISSING] Table not found: {table}")

## Null checks - existing tables

In [0]:
spark.sql(f"""
SELECT
    'gold_regional_operations' AS table_name,
    SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END)        AS null_event_date,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END)            AS null_region,
    SUM(CASE WHEN avg_price_eur_mwh IS NULL THEN 1 ELSE 0 END) AS null_avg_price,
    SUM(CASE WHEN operational_risk IS NULL THEN 1 ELSE 0 END)  AS null_operational_risk
FROM {gold_regional_operations_table}
""").show()

spark.sql(f"""
SELECT
    'gold_market_volatility' AS table_name,
    SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END)        AS null_event_date,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END)            AS null_region,
    SUM(CASE WHEN avg_price_eur_mwh IS NULL THEN 1 ELSE 0 END) AS null_avg_price,
    SUM(CASE WHEN volatility_band IS NULL THEN 1 ELSE 0 END)   AS null_volatility_band
FROM {gold_market_volatility_table}
""").show()

## Null checks - new tables

In [0]:
spark.sql(f"""
SELECT
    'gold_market_summary' AS table_name,
    SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END)        AS null_event_date,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END)            AS null_region,
    SUM(CASE WHEN avg_price_eur_mwh IS NULL THEN 1 ELSE 0 END) AS null_avg_price,
    SUM(CASE WHEN volatility_band IS NULL THEN 1 ELSE 0 END)   AS null_volatility_band
FROM {gold_market_summary_table}
""").show()

spark.sql(f"""
SELECT
    'gold_grid_summary' AS table_name,
    SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END)               AS null_event_date,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END)                   AS null_region,
    SUM(CASE WHEN operational_risk IS NULL THEN 1 ELSE 0 END)         AS null_operational_risk,
    SUM(CASE WHEN critical_event_count IS NULL THEN 1 ELSE 0 END)     AS null_critical_event_count
FROM {gold_grid_summary_table}
""").show()

spark.sql(f"""
SELECT
    'gold_weather_impact' AS table_name,
    SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END)              AS null_event_date,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END)                  AS null_region,
    SUM(CASE WHEN avg_temperature_c IS NULL THEN 1 ELSE 0 END)       AS null_avg_temperature,
    SUM(CASE WHEN weather_impact_score IS NULL THEN 1 ELSE 0 END)    AS null_weather_impact_score
FROM {gold_weather_impact_table}
""").show()

spark.sql(f"""
SELECT
    'gold_operational_kpi' AS table_name,
    SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END)                        AS null_event_date,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END)                            AS null_region,
    SUM(CASE WHEN operational_risk IS NULL THEN 1 ELSE 0 END)                  AS null_operational_risk,
    SUM(CASE WHEN pct_high_risk_days_cumulative IS NULL THEN 1 ELSE 0 END)     AS null_pct_high_risk
FROM {gold_operational_kpi_table}
""").show()

## Negative value checks

In [0]:
spark.sql(f"""
SELECT 'avg_price_eur_mwh'             AS column_name, COUNT(*) AS negative_count FROM {gold_regional_operations_table} WHERE avg_price_eur_mwh < 0
UNION ALL
SELECT 'total_volume_mwh',             COUNT(*) FROM {gold_regional_operations_table} WHERE total_volume_mwh < 0
UNION ALL
SELECT 'critical_event_count',         COUNT(*) FROM {gold_regional_operations_table} WHERE critical_event_count < 0
UNION ALL
SELECT 'weather_impact_score',         COUNT(*) FROM {gold_regional_operations_table} WHERE weather_impact_score < 0
UNION ALL
SELECT 'grid_critical_event_count',    COUNT(*) FROM {gold_grid_summary_table} WHERE critical_event_count < 0
UNION ALL
SELECT 'grid_total_events',            COUNT(*) FROM {gold_grid_summary_table} WHERE total_events < 0
UNION ALL
SELECT 'weather_avg_wind_speed_kmh',   COUNT(*) FROM {gold_weather_impact_table} WHERE avg_wind_speed_kmh < 0
UNION ALL
SELECT 'weather_total_precipitation',  COUNT(*) FROM {gold_weather_impact_table} WHERE total_precipitation_mm < 0
""").show()

## Distribution checks

In [0]:
spark.sql(f"""
SELECT operational_risk, COUNT(*) AS count
FROM {gold_regional_operations_table}
GROUP BY operational_risk ORDER BY count DESC
""").show()

spark.sql(f"""
SELECT volatility_band, COUNT(*) AS count
FROM {gold_market_volatility_table}
GROUP BY volatility_band ORDER BY count DESC
""").show()

spark.sql(f"""
SELECT operational_risk, COUNT(*) AS count
FROM {gold_grid_summary_table}
GROUP BY operational_risk ORDER BY count DESC
""").show()

spark.sql(f"""
SELECT dominant_wind_class, COUNT(*) AS count
FROM {gold_weather_impact_table}
GROUP BY dominant_wind_class ORDER BY count DESC
""").show()

## Business rule checks - valid categorical values

In [0]:
valid_risk   = {"HIGH", "MEDIUM", "LOW"}
valid_band   = {"HIGH", "MEDIUM", "LOW"}

for table, col, valid in [
    (gold_regional_operations_table, "operational_risk", valid_risk),
    (gold_grid_summary_table,        "operational_risk", valid_risk),
    (gold_market_volatility_table,   "volatility_band",  valid_band),
    (gold_market_summary_table,      "volatility_band",  valid_band),
]:
    actual = {
        row[col] for row in spark.sql(f"SELECT DISTINCT {col} FROM {table}").collect()
        if row[col] is not None
    }
    invalid = actual - valid
    if invalid:
        log.warning(f"[RULE] {table}.{col} has invalid values: {invalid}")
    else:
        log.info(f"[RULE] {table}.{col} OK - values: {actual}")

## Summary check - cross table consistency

In [0]:
ops_count     = spark.table(gold_regional_operations_table).count()
vol_count     = spark.table(gold_market_volatility_table).count()
market_count  = spark.table(gold_market_summary_table).count()
grid_count    = spark.table(gold_grid_summary_table).count()
weather_count = spark.table(gold_weather_impact_table).count()
kpi_count     = spark.table(gold_operational_kpi_table).count()

print(f"[SUMMARY] {gold_regional_operations_table}: {ops_count} rows")
print(f"[SUMMARY] {gold_market_volatility_table}:   {vol_count} rows")
print(f"[SUMMARY] {gold_market_summary_table}:      {market_count} rows")
print(f"[SUMMARY] {gold_grid_summary_table}:        {grid_count} rows")
print(f"[SUMMARY] {gold_weather_impact_table}:      {weather_count} rows")
print(f"[SUMMARY] {gold_operational_kpi_table}:     {kpi_count} rows")

if ops_count != vol_count:
    log.warning(f"[SUMMARY] Row count mismatch: regional_operations={ops_count} vs market_volatility={vol_count}")
else:
    log.info("[SUMMARY] Row counts consistent: regional_operations and market_volatility match.")

if grid_count != weather_count:
    log.warning(f"[SUMMARY] Row count mismatch: grid_summary={grid_count} vs weather_impact={weather_count}")
else:
    log.info("[SUMMARY] Row counts consistent: grid_summary and weather_impact match.")

## Retrieve taskValue from gold outputs task

In [0]:
try:
    expected_count = dbutils.jobs.taskValues.get(taskKey="gold_outputs", key="gold_row_count")
    actual_count   = str(ops_count)
    print(f"Expected rows from gold_outputs task: {expected_count}")
    print(f"Actual rows in gold table:            {actual_count}")
    if expected_count != actual_count:
        log.warning(f"[TASKVALUE] Row count mismatch: expected {expected_count}, got {actual_count}")
    else:
        log.info("[TASKVALUE] Row count matches.")
except Exception:
    log.info("[TASKVALUE] Not running in a job context - skipping taskValues check.")